In [2]:
import os
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt


def psth_laser_and_offset(spikes_path: str,laser_times_path: str,save_folder: str,laser_type: str, offset_s: float = 15.0,        
        bin_size: float = 0.005, time_window: tuple[float, float] = (-1.0, 1.5),
        laser_duration: float = 0.5,type_file: str = "pdf",
):
    """
    Open-field PSTHs for every unit:
    """
    # ------------------------------------------------------------------ I/O --
    spikes = np.load(spikes_path, allow_pickle=True)
    mat    = sio.loadmat(laser_times_path)

    try:
        laser_times = mat[laser_type][0, 0]['Ts'].flatten()
    except (KeyError, IndexError, AttributeError) as e:
        raise ValueError(f"Could not locate '{laser_type}' in {laser_times_path}") from e

    if laser_times.size == 0:
        raise ValueError("No laser timestamps found")

    units = np.unique(spikes['unit_index'])

    # make output dirs 
    laser_dir   = os.path.join(save_folder, "laser")
    control_dir = os.path.join(save_folder, f"control_offset{int(offset_s)}s")
    for d in (laser_dir, control_dir):
        os.makedirs(d, exist_ok=True)

    # helpers 
    def align(spk, refs):
        return [spk[(spk >= r + time_window[0]) &
                    (spk <= r + time_window[1])] - r
                for r in refs]

    def psth(aligned):
        edges    = np.arange(time_window[0], time_window[1] + bin_size, bin_size)
        centers  = edges[:-1] + bin_size / 2
        if not aligned:
            return centers, np.zeros_like(centers), np.zeros_like(centers), 0
        counts   = np.vstack([np.histogram(a, bins=edges)[0] for a in aligned])
        rates    = counts / bin_size
        return centers, rates.mean(0), rates.std(0) / np.sqrt(rates.shape[0]), rates.shape[0]

    def plot(xs, mean, sem, ntr, unit_id, title, out_path):
        plt.figure(figsize=(10, 6))
        plt.plot(xs, mean, lw=2)
        plt.fill_between(xs, np.maximum(mean - sem, 0), mean + sem, alpha=.25)
        plt.axvspan(0, laser_duration, color='#ADD8E6', alpha=0.3, label='Highlight')
        plt.axvline(0, color='k', ls='--', lw=1)
        plt.xlabel('Time from reference (s)')
        plt.ylabel('Firing rate (Hz)')
        plt.title(f'{title} | Unit {unit_id} | n={ntr}')
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, format=type_file)
        plt.close()

    #  iterate
    for u in units:
        print(f'Unit {u}: processing')
        spk_sec = spikes['sample_index'][spikes['unit_index'] == u] / 40_000.0

        #  real laser
        xs, du, do, n = psth(align(spk_sec, laser_times))
        plot(xs, du, do, n, u, 'Laser aligned', 
             os.path.join(laser_dir, f'laser_{u}.{type_file}'))

        #  offset control
        xs_c, du_c, do_c, n_c = psth(align(spk_sec, laser_times + offset_s))
        plot(xs_c, du_c, do_c, n_c, u, f'Laser+{offset_s:g}s aligned', 
             os.path.join(control_dir, f'control_{u}.{type_file}'))

    print(f'Finished — figures in:\n  {laser_dir}\n  {control_dir}')


In [1]:
from pathlib import Path

session_id = "Rec_Upstream_SNr_4_250622_mixed_2point5_5_10mW_100ms_0delay_062225001"

root = Path("E:/")           
base_folder = root / "Paolo" / "temp_recordings" \
                    / "Upstream_SNr_4"  / session_id

print(base_folder)         
print(base_folder.exists()) 


E:\Paolo\temp_recordings\Upstream_SNr_4\Rec_Upstream_SNr_4_250622_mixed_2point5_5_10mW_100ms_0delay_062225001
True


In [12]:
from pathlib import Path

session_id = "Rec_Upstream_SNr_4_250613_500delay_500ms_2point5mW_061325001"

#Rec_Upstream_SNr_4_250614_0delay_500ms_2point5mW_Licking_061425001
#Rec_Upstream_SNr_4_250615_500delay_500ms_5mW_Licking_061525001
#Rec_Upstream_SNr_4_250616_500ms_5mW_061625001
#Rec_Upstream_SNr_4_250617_mixed1_2point5_5mW_500ms_500delay_061725001
#Rec_Upstream_SNr_4_250618_mixed1_2point5_5mW_500ms_0delay_061825001
#Rec_Upstream_SNr_4_250619_5mW_100ms_0delay_061925001

#root = Path("N:/")        
#base_folder = root / "MICROSCOPE" / "Paolo" / "Recordings" \
#                    / "Rec_Upstream_SNr_4" / "Rec_Upstream_SNr_4_SI" / session_id


root = Path("E:/")        
base_folder = root / "Paolo" / "temp_recordings" \
                    / "Upstream_SNr_4"  / session_id

print(base_folder)         
print(base_folder.exists()) 

bin_size = 0.02
laser_duration = 0.5
control_offset = 15.0  # seconds
laser_type = "opto_laser_evt14"
output_folder = "open_field"

# build parameters
spikes_path = rf"{base_folder}\spikeinterface\analyzer\sorting\spikes.npy"
laser_timestamps_path = rf"{base_folder}\{session_id}.mat"
output_folder_psth_1_path = base_folder / "spikeinterface" / output_folder


spikes_path           = base_folder / "spikeinterface" / "analyzer" / "sorting" / "spikes.npy"
laser_timestamps_path = base_folder / f"{session_id}.mat"
output_folder_psth_1  = base_folder / "spikeinterface" / output_folder

psth_laser_and_offset(
    spikes_path=spikes_path,
    laser_times_path=laser_timestamps_path,
    save_folder=output_folder_psth_1_path,
    laser_type=laser_type,  
    offset_s=control_offset,               
    bin_size=bin_size,
    time_window=(-1, 2),
    laser_duration=laser_duration,
    type_file='pdf',
)


E:\Paolo\temp_recordings\Upstream_SNr_4\Rec_Upstream_SNr_4_250613_500delay_500ms_2point5mW_061325001
True
Unit 0: processing
Unit 1: processing
Unit 2: processing
Unit 3: processing
Unit 4: processing
Unit 5: processing
Unit 6: processing
Unit 7: processing
Unit 8: processing
Unit 9: processing
Unit 10: processing
Unit 11: processing
Unit 12: processing
Unit 13: processing
Unit 14: processing
Unit 15: processing
Unit 16: processing
Unit 17: processing
Unit 18: processing
Unit 19: processing
Unit 20: processing
Unit 21: processing
Unit 22: processing
Unit 23: processing
Unit 24: processing
Unit 25: processing
Unit 26: processing
Unit 27: processing
Unit 28: processing
Unit 29: processing
Unit 30: processing
Unit 31: processing
Unit 32: processing
Unit 33: processing
Unit 34: processing
Unit 35: processing
Unit 36: processing
Unit 37: processing
Unit 38: processing
Unit 39: processing
Unit 40: processing
Unit 41: processing
Unit 42: processing
Unit 43: processing
Unit 44: processing
Unit